# Generate Head Boundary Conditions for 2D transect ATS - Naches-0

Head extracted from Zhi's 3D ATS simulation for Naches

- file `global/cfs/cdirs/m1800/naches_run2_share/Naches-2`
- Information
    - "Time": from 11224 to 16059; unit is day;
        - 11400 = 85+365x31 --> 2011.03.26
        - 12631 = 221+365x34 --> 2014.08.09

Output of this script

- constant head at the starting and end points -> to drive the run0
- head at a typical year at the starting and end points -> to drive the run1
- transient head

**File History**

update 2026/1/29
- previously, it was a two-step extraction
    - 1. extract point water head on nersc
    - 2. process cyclic spinup and transient
- now, merge two steps into this notebook
    - merge with `10-Projects/2025-RCSFA-HillslopeFire/MaterialsData/OakCreek_from_sundar/ats_WTD_pressure_based_Aug1_2024.ipynb` on NERSC

update 2025/10/13
- update `config.json`. Mainly revise the model run pipeline.

update 2025/8/32
- add `config.json`

update 2025/8/7
- correct site name to NF01
- a cleaned version putting all input data and notebooks together

In [ ]:
%load_ext autoreload
%autoreload 2

## config Parameters and data sources

In [ ]:
# Parameters cell
import json
with open('config.json', 'r') as f:
    config = json.load(f)
watershed_name = config['watershed_name']
hucs           = [config['hucs']]
site_name      = config['site_name']

# simulation control
start_year_spinup         = config['start_year_spinup']
end_year_spinup           = config['end_year_spinup']
nyears_steadystate_spinup = config['nyears_steadystate_spinup']
nyears_cyclic_spinup      = config['nyears_cyclic_spinup']
start_year_transient      = config['start_year_transient']
end_year_transient        = config['end_year_transient']

In [ ]:
outputs={}

## extract point raw data from 3D ATS simulation

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5

import scipy.signal
from datetime import datetime, timedelta
import pandas as pd

import ats_xdmf as xdmf
import time
import random
import pandas
import os

from scipy.io import loadmat
import shapely
from shapely.geometry import Point, LineString, Polygon, box, mapping
import geopandas as gpd

In [ ]:
## get stream network
# Essential imports for watershed workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.plot
import pyproj

# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)

# Set up watershed workflow CRS (DayMet CRS)
crs_daymet = watershed_workflow.crs.daymet_crs()

# Set up sources
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']

# Parameters for river extraction
hucs_config = [config['hucs']]
ignore_small_rivers = 2
prune_by_area_fraction = 0.0

# Get HUC12 list
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
    return huc12_list

hucs = get_huc12(hucs_config)
huc_level = 12

print(f"Processing HUCs: {hucs[:5]}...")

# Load watershed HUCs
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs_daymet)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

# Download/collect the river network
print("Downloading river network...")
_, reaches = watershed_workflow.get_reaches(sources['hydrography'], hucs[0], 
                                            watershed.exterior(), crs_daymet, crs_daymet,
                                            in_network=True, properties=True)

# Construct river network
rivers = watershed_workflow.construct_rivers(reaches, method='hydroseq',
                                             ignore_small_rivers=ignore_small_rivers,
                                             prune_by_area=prune_by_area_fraction * watershed.exterior().area * 1.e-6,
                                             remove_diversions=True,
                                             remove_braided_divergences=True)

print(f"Number of rivers: {len(rivers)}")

### Config 3D ATS-flow results

In [ ]:
# Define the output model directory, whre ats_vis data (.h5) are located
model_dir = '/global/cfs/cdirs/m1800/naches_run2_share/Naches-2'

# Define the Parameters AND verify with the XML
rho = 997 # density of water, kg m^-3
g = 9.80665 # gravity, m s^-2
patm = 101325 # atmopsheric pressure, Pascals

# Define raw output, and skip raw point data extraction if detect it
outputs['tmp_BChead_raw'] = f'../data-processed/{site_name}/tmp_bc_startend_raw.naches-2.h5'

# Check if file exists
if os.path.exists(outputs['tmp_BChead_raw']):
    flag_atsptextract = False
    print(f"File {outputs['tmp_BChead_raw']} exists, skipping extraction.")
else:
    flag_atsptextract = True
    print(f"File {outputs['tmp_BChead_raw']} not found, will extract data.")

flag_atsptextract = True

### Prepare hillslope shape

In [ ]:
# load hillslope geometry from mat file generated in "1-full_workflow_OakCreek.ipynb"
#site_name = 'NF01'
#meshsize_nx = 100

m2_mat_filename =  f'../data-processed/{site_name}/m2_coords_{site_name}.mat'
loaded_data = loadmat(m2_mat_filename)
meshsize_nx = loaded_data['meshsize_nx'].flatten()[0]
#dzs_soil  = loaded_data['dzs_soil'].flatten()
#dzs_geo   = loaded_data['dzs_geo'].flatten()
#m2_coords = loaded_data['m2_coords']
loaded_gdf_dict = loaded_data['gdf_data']
gdf_reloaded = pd.DataFrame({
    'lon': loaded_gdf_dict['lon'][0, 0].flatten(),
    'lat': loaded_gdf_dict['lat'][0, 0].flatten(),
    'h_distance': loaded_gdf_dict['h_distance'][0, 0].flatten(),
    'elevation': loaded_gdf_dict['elevation'][0, 0].flatten()
})
geometry = [Point(xy) for xy in zip(gdf_reloaded['lon'], gdf_reloaded['lat'])]
hillslope_gdf = gpd.GeoDataFrame(gdf_reloaded, geometry=geometry)

# create hillslope polygon - already a shapely object, no conversion needed
xsec_plg = Polygon([hillslope_gdf.geometry[i] for i in range(hillslope_gdf.shape[0])])
xsec_plg_dict = {"type": "Feature", "id":0, "properties":{}, "geometry": mapping(xsec_plg)}
# In watershed-workflow 2.0, use the shapely Polygon directly
xsec_plg_dict_shply = xsec_plg

# convert to latlon crs using warp.shply() in v2.0
# reproj_xsec_plg = watershed_workflow.warp.shply(xsec_plg, crs_daymet, crs_latlon)

In [ ]:
hillslope_gdf

### Load surface/subsurface h5 files

In [ ]:
## a function to estimate WTD based on pressure

def get_ats_wtd_pressurebased(pressure_subsurface,visfile_surface, visfile_subsurface):
    # visfile_subsurface.centroids.shape -> (n_surface, 14, 3); 14 is soil layers, bottom-top
    iz_coord = visfile_subsurface.centroids[:,:,-1]
    
    ### Find Equivalent Surface and Subsurface ID based on cell centroids
    # take some care on the rounding of coordinates, can cause errors
    surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
    subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
    # [remarks] prepare to perform a pairwise compare [n_surface,1,2] with [1,n_surface,2] -> returns [n_surface, n_surface]
    surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
    subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
    matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
    surface_indices, subsurface_indices = np.nonzero(matches)
    surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))
    assert surface_subsurface_IDs[:,1].shape == visfile_surface.centroids[:,1].shape, f"Shape mismatch: change the round precision in the surface/subsurface centroid coordinates"
    
    ### Estimate WTD based on pressure
    # pressure head
    ih = (pressure_subsurface - patm) / (rho * g) # dim=(time, n_surface, 14)
    mask = ih > 0
    first_false_idx = (~mask).argmax(axis=-1) # dim=(time, n_surface)
    max_len = mask.shape[-1]
    indices = np.arange(max_len)
    mask2 = indices < first_false_idx[..., np.newaxis]
    all_true_rows = (first_false_idx == 0)
    mask2[all_true_rows] = True
    sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
    sat_idx = mask2.shape[-1] - 1 - sat_idx
    sat_idx[~mask2.any(axis=-1)] = 0
    # WTD elevation
    # [remark] iH_rev=part1+part2; part1 is the relative distance between water table and the first saturated subsurface cell
    iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx] 
    first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
    # WTD (from surface)
    head_pressure_based = -(iz_coord[:,-1]+first_depth_to_centroid - iH_rev) #[added by Yi]
    wtdep_pressure_based = np.maximum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    pwdep_pressure_based = - np.minimum(iz_coord[:,-1]+first_depth_to_centroid - iH_rev, 0)
    assert pressure_subsurface[:,:,1].shape == wtdep_pressure_based.shape, f"Shape mismatch: Error in WTD calculation"
    
    ### Re-arrange WTD in accordance to surface IDs
    # As pressure obtained from subsurface does not follow surface ID, re-arrange this:
    head_rearranged  = head_pressure_based[:, subsurface_indices]
    wtdep_rearranged = wtdep_pressure_based[:, subsurface_indices]
    pwdep_rearranged = pwdep_pressure_based[:, subsurface_indices]
    
    return surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged

In [ ]:
if flag_atsptextract:
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    print(visfile_surface.times)

    tmp_year = 2026 # a standard year
    day_number0 = int(visfile_surface.times[0])
    date0 = datetime.strptime(f'{tmp_year}-{day_number0%365}', '%Y-%j')
    print(f"t0: {day_number0}={day_number0%365}+365x{day_number0//365}")
    print(f"t0: {day_number0%365} is {date0.strftime('%B %d')}")
    
    day_number1 = int(visfile_surface.times[-1])
    date1 = datetime.strptime(f'{tmp_year}-{day_number1%365}', '%Y-%j')
    print(f"t1: {day_number1}={day_number1%365}+365x{day_number1//365}")
    print(f"t1: {day_number1%365} is {date1.strftime('%B %d')}")
    
else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # subsurface
    start = time.time()
    visfile_subsurface = xdmf.VisFile(directory=model_dir,
                                      filename="ats_vis_data.h5", 
                                      mesh_filename="ats_vis_mesh.h5")
    visfile_subsurface.loadMesh(columnar=True)
    end = time.time()
    print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")
    
    # surface
    start = time.time()
    visfile_surface = xdmf.VisFile(directory=model_dir,
                                   domain="surface", 
                                   filename="ats_vis_surface_data.h5" , 
                                   mesh_filename="ats_vis_surface_mesh.h5")
    visfile_surface.loadMesh()
    end = time.time()
    print(f"Time cost for surface VisFile: {end - start:.6f} seconds")
    
    # subsurface pressure (time, xy-space and soil columns)
    start = time.time()
    pressure_subsurface = visfile_subsurface.getArray('pressure')
    end = time.time()
    print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")
    
    # get the WTD (based on surface ATS ID)
    start = time.time()
    surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged = get_ats_wtd_pressurebased(pressure_subsurface=pressure_subsurface,
                                                                                    visfile_surface=visfile_surface, visfile_subsurface= visfile_subsurface)
    end = time.time()
    print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

else:
    print("Skipping: data already extracted")

### Extract water head at the start & end points of a 2D transect

In [ ]:
if flag_atsptextract:
    surface_x_coord = visfile_surface.centroids[:,0]
    surface_y_coord = visfile_surface.centroids[:,1]
    
    # print(surface_x_coord)
    # print(surface_x_coord.shape)
    # print(surface_y_coord)
    # print(surface_y_coord.shape)
    
    start_coords = (gdf_reloaded['lon'].iloc[0], gdf_reloaded['lat'].iloc[0])
    end_coords   = (gdf_reloaded['lon'].iloc[-1], gdf_reloaded['lat'].iloc[-1])
    
    # Get times from surface
    surface_times = visfile_surface.times
    print(f"Number of timesteps: {len(surface_times)}")
    print(surface_times)

else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # Compute Euclidean distances
    dist_start = np.sqrt((surface_x_coord - start_coords[0])**2 + (surface_y_coord - start_coords[1])**2)
    dist_end = np.sqrt((surface_x_coord - end_coords[0])**2 + (surface_y_coord - end_coords[1])**2)
    
    # Get the index of the closest point
    start_index = np.argmin(dist_start)
    end_index = np.argmin(dist_end)
    
    # Get the minimum distances
    min_dist_start = dist_start[start_index]
    min_dist_end = dist_end[end_index]
    
    # Print results
    print(f"Index for start_coords: {start_index}, Minimum distance: {min_dist_start}")
    print(f"Index for end_coords: {end_index}, Minimum distance: {min_dist_end}")

else:
    print("Skipping: data already extracted")

In [ ]:
if flag_atsptextract:
    # outputs['tmp_BChead_raw'] = f'../data-processed/{site_name}/tmp_bc_startend_raw.h5'
    # Extract data based on computed indices
    startpt_head = head_rearranged[:, start_index]
    endpt_head = head_rearranged[:, end_index]
    
    # # Write to HDF5
    # with h5.File(outputs['tmp_BChead_raw'], "w") as hdf:
    #     hdf.create_dataset("Time", data=surface_times)
    #     hdf.create_dataset("startpt_head", data=startpt_head)
    #     hdf.create_dataset("endpt_head", data=endpt_head)
    
    # print(f"Data successfully written to {outputs['tmp_BChead_raw']}")

else:
    print("Skipping: data already extracted")

## Check neighbour triangles of the river cell

In [ ]:
# Define N
N = 7  # Number of closest points you want

# Compute Euclidean distances
dist_start = np.sqrt((surface_x_coord - start_coords[0])**2 + (surface_y_coord - start_coords[1])**2)
dist_end = np.sqrt((surface_x_coord - end_coords[0])**2 + (surface_y_coord - end_coords[1])**2)

# Get the indices of the N closest points
start_indices = np.argpartition(dist_start, N)[:N]
end_indices = np.argpartition(dist_end, N)[:N]

# Sort them by distance (optional, for cleaner output)
start_indices = start_indices[np.argsort(dist_start[start_indices])]
end_indices = end_indices[np.argsort(dist_end[end_indices])]

# Get the corresponding distances
start_distances = dist_start[start_indices]
end_distances = dist_end[end_indices]

# Print results
print(f"Top {N} closest points to start_coords:")
for i, (idx, dist) in enumerate(zip(start_indices, start_distances), 1):
    print(f"  {i}. Index: {idx}, Distance: {dist:.4f}")

print(f"\nTop {N} closest points to end_coords:")
for i, (idx, dist) in enumerate(zip(end_indices, end_distances), 1):
    print(f"  {i}. Index: {idx}, Distance: {dist:.4f}")

# To use just the closest point (like before)
start_index = start_indices[0]
end_index = end_indices[0]

In [ ]:
time = visfile_surface.times
print(time)

### mesh plot

In [ ]:
import matplotlib.collections as mc

# Get mesh info directly
etype, coords, conn = xdmf.meshXYZ(model_dir, "ats_vis_surface_mesh.h5")

# Create polygon coordinates for triangles (use x, y coordinates only)
polygon_coords = [coords[c][:, :2] for c in conn]

# Create polygon collection
polygons = mc.PolyCollection(polygon_coords, edgecolor='k', cmap='Blues', linewidth=0.5)

# Get some data to visualize (e.g., pressure at first cycle)
# print("Available variables:")
# print(list(visfile_surface.d.keys()))
data = visfile_surface.get('surface-ponded_depth', visfile_surface.cycles[0])
polygons.set_array(data)
polygons.set_clim(vmin=0, vmax=0.01)  # Set specific range


# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.add_collection(polygons)

# Highlight the polygon of the closest point to end_coords
closest_idx = end_indices[0]
highlighted_polygon = mc.PolyCollection([polygon_coords[closest_idx]], 
                                       edgecolor='red', 
                                       facecolors='none', 
                                       linewidth=3)
ax.add_collection(highlighted_polygon)

# add rivers
watershed_workflow.plot.rivers(rivers, crs_daymet, ax=ax, colors='gold', linewidth=1.0)

# Zoom to region around start/end coordinates with buffer
x_coords = [start_coords[0], end_coords[0]]
y_coords = [start_coords[1], end_coords[1]]
x_range = max(x_coords) - min(x_coords)
y_range = max(y_coords) - min(y_coords)
buffer = 0.1  # 10% buffer
ax.set_xlim(min(x_coords) - 1.5 * x_range, max(x_coords) + 0.4 * x_range)
ax.set_ylim(min(y_coords) - 4 * y_range, max(y_coords) + 4 * y_range)

ax.set_aspect('equal')
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
plt.colorbar(polygons, ax=ax, label='surface-ponded_depth')

plt.title(f'Surface Mesh - Cycle {visfile_surface.cycles[0]}')

ax.plot(*start_coords, 'rx', markersize=10, label='Start')
ax.plot(*end_coords, 'ro', markersize=10, label='End')
ax.plot(surface_x_coord[closest_idx], surface_y_coord[closest_idx], 
        'go', markersize=8, label='Closest centroid to End')

# Plot top N closest points to end_coords
for i, idx in enumerate(end_indices):
    x, y = surface_x_coord[idx], surface_y_coord[idx]
    if i == 0:
        ax.plot(x, y, 'yo', markersize=8, label=f'Top {N} closest to End')
    else:
        ax.plot(x, y, 'yo', markersize=8)

    # Add rank number as text annotation
    ax.text(x, y, str(i+1), fontsize=10, ha='center', va='center', 
            color='red', fontweight='bold',
            bbox=dict(boxstyle='circle,pad=0.3', facecolor='white', edgecolor='red', alpha=0.8))

plt.tight_layout()
plt.show()

In [ ]:
# Extract head data for top N closest points to end_coords
endpt_heads = [head_rearranged[:, idx] for idx in end_indices]

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Define colors for each line
colors = plt.cm.tab10(np.linspace(0, 1, N))

# Plot top 1 with solid line, rest with dashed lines
for i, (idx, head_data) in enumerate(zip(end_indices, endpt_heads)):
    if i == 0:
        # Top 1: solid line
        ax.plot(time, head_data, color=colors[i], linewidth=2, 
                label=f'Closest (idx={idx}, dist={end_distances[i]:.2f}m)')
    else:
        # Others: dashed lines
        ax.plot(time, head_data, color=colors[i], linewidth=1.5, linestyle='--',
                label=f'#{i+1} (idx={idx}, dist={end_distances[i]:.2f}m)')

ax.set_xlabel('Time [days]')
ax.set_ylabel('Head [m]')
ax.set_title(f'Head time series for top {N} closest points to end_coords')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Extract ponded water depth data for top N closest points to end_coords
endpt_pwdeps = [pwdep_rearranged[:, idx] for idx in end_indices]

# Plot
fig, ax = plt.subplots(figsize=(12, 6))

# Define colors for each line
colors = plt.cm.tab10(np.linspace(0, 1, N))

# Plot top 1 with solid line, rest with dashed lines
for i, (idx, pwdep_data) in enumerate(zip(end_indices, endpt_pwdeps)):
    if i == 0:
        # Top 1: solid line
        ax.plot(time, pwdep_data, color=colors[i], linewidth=2, 
                label=f'Closest (idx={idx}, dist={end_distances[i]:.2f}m)')
    else:
        # Others: dashed lines
        ax.plot(time, pwdep_data, color=colors[i], linewidth=1.5, linestyle='--',
                label=f'#{i+1} (idx={idx}, dist={end_distances[i]:.2f}m)')

ax.set_xlabel('Time [days]')
ax.set_ylabel('Head [m]')
ax.set_title(f'Ponded water depth time series for top {N} closest points to end_coords')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot pwdep_data from 2nd to 7th closest vs 1st closest
fig, ax = plt.subplots(figsize=(8, 6))

# Get data for the closest point (reference)
closest_pwdep = endpt_pwdeps[0]

# Define colors for each line
colors = plt.cm.tab10(np.linspace(0, 1, N))

# Plot 2nd to 7th closest against 1st closest
for i in range(1, N):
    ax.scatter(closest_pwdep, endpt_pwdeps[i], color=colors[i], alpha=0.5, s=10,
              label=f'#{i+1} (idx={end_indices[i]}, dist={end_distances[i]:.2f}m)')

# Add 1:1 reference line
min_val = min(closest_pwdep.min(), min([ep.min() for ep in endpt_pwdeps[1:]]))
max_val = max(closest_pwdep.max(), max([ep.max() for ep in endpt_pwdeps[1:]]))
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, label='1:1 line')

ax.set_xlabel(f'Closest point ponded depth [m] (idx={end_indices[0]})')
ax.set_ylabel('Other points ponded depth [m]')
ax.set_title(f'Ponded water depth: 2nd-7th closest vs 1st closest to end_coords')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Extract surface-vis-file based ponded water depth
# compare with the subsurface-vis-file based ponded water depth

# print("Available variables:")
# print(list(visfile_surface.d.keys()))
surface_ponded_depth = visfile_surface.getArray('surface-ponded_depth')
closest_pwdep_surface_vis = surface_ponded_depth[:, end_indices[0]]


# Plot pwdep_data from 2nd to 7th closest vs 1st closest
fig, ax = plt.subplots(figsize=(8, 6))

# Get data for the closest point (reference)
closest_pwdep = endpt_pwdeps[0]

ax.scatter(closest_pwdep, closest_pwdep_surface_vis, color='blue', alpha=0.5, s=10)

# Add 1:1 reference line
min_val = min(closest_pwdep.min(), min([ep.min() for ep in endpt_pwdeps[1:]]))
max_val = max(closest_pwdep.max(), max([ep.max() for ep in endpt_pwdeps[1:]]))
ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, label='1:1 line')

ax.set_xlabel(f'Ponded water depth based on subsurface pressure')
ax.set_ylabel('Other points ponded depth [m]')
ax.set_title(f'Ponded water depth from surface vis file')
#ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## Find river cells and river segments in river cells

In [ ]:
# Calculate percentage of positive values for each grid point
n_time, n_space = head_rearranged.shape
positive_percentages = np.zeros(n_space)

for idx in range(n_space):
    n_positive = np.sum(head_rearranged[:, idx] > 0)
    positive_percentages[idx] = (n_positive / n_time) * 100

# Plot histogram
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(positive_percentages, bins=50, edgecolor='black', alpha=0.7)
ax.set_xlabel('Percentage of positive head values (%)')
ax.set_ylabel('Number of grid points')
ax.set_title(f'Distribution of positive head percentage across {n_space} grid points')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Statistics:")
print(f"  Total grid points: {n_space}")
print(f"  Total timesteps: {n_time}")
print(f"  Min positive percentage: {np.min(positive_percentages):.2f}%")
print(f"  Max positive percentage: {np.max(positive_percentages):.2f}%")
print(f"  Grid points with >50% positive: {np.sum(positive_percentages > 50)}")
print(f"  Grid points with 100% positive: {np.sum(positive_percentages == 100)}")

In [ ]:
# Identify grid points with 100% positive head
fully_positive_mask = positive_percentages == 100
fully_positive_indices = np.where(fully_positive_mask)[0]

print(f"Grid points with 100% positive head: {np.sum(fully_positive_mask)} out of {n_space}")

# Create mesh plot highlighting 100% positive polygons
import matplotlib.collections as mc

# Get mesh info directly
etype, coords, conn = xdmf.meshXYZ(model_dir, "ats_vis_surface_mesh.h5")

# Create polygon coordinates for triangles (use x, y coordinates only)
polygon_coords = [coords[c][:, :2] for c in conn]

# Create polygon collection for base mesh
polygons = mc.PolyCollection(polygon_coords, edgecolor='k', cmap='Blues', linewidth=0.5, alpha=0.7)

# Get ponded depth data to visualize
data = visfile_surface.get('surface-ponded_depth', visfile_surface.cycles[0])
polygons.set_array(data)
polygons.set_clim(vmin=0, vmax=0.01)

# Create highlighted polygon collection for 100% positive head
highlighted_polygons_coords = [polygon_coords[i] for i in fully_positive_indices]
highlighted_polygons = mc.PolyCollection(highlighted_polygons_coords, 
                                        edgecolor='red', 
                                        facecolors='none', 
                                        linewidth=0.5,
                                        alpha=0.7)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
ax.add_collection(polygons)
ax.add_collection(highlighted_polygons)

# Add rivers
watershed_workflow.plot.rivers(rivers, crs_daymet, ax=ax, colors='blue', linewidth=1.0)

# Set plot limits to show the full mesh
ax.set_xlim(coords[:, 0].min(), coords[:, 0].max())
ax.set_ylim(coords[:, 1].min(), coords[:, 1].max())

ax.set_aspect('equal')
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
plt.colorbar(polygons, ax=ax, label='surface-ponded_depth [m]')

plt.title(f'Surface Mesh with 100% Positive Head Polygons Highlighted\nCycle {visfile_surface.cycles[0]} | {np.sum(fully_positive_mask)}/{n_space} polygons with 100% positive head')

# Add legend for highlighted polygons
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='none', edgecolor='red', linewidth=1, label='100% positive head')]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Define "river cells" as triangles with 100% positive head
river_cell_indices = fully_positive_indices
print(f"Number of river cells: {len(river_cell_indices)}")

# Create polygons from river cells for intersection analysis
from shapely.geometry import Polygon as ShapelyPolygon, LineString as ShapelyLineString
river_cell_polygons = []
for idx in river_cell_indices:
    poly_coords = polygon_coords[idx]
    river_cell_polygons.append(ShapelyPolygon(poly_coords))

# Identify individual line segments that overlap with river cells
# Each LineString is made of multiple line segments (consecutive coordinate pairs)
line_segments_in_cells = []
line_segments_in_cells_info = []  # Store (river_idx, segment_idx, line_seg_idx) tuples

print(f"Total River objects: {len(rivers)}")
print(f"Examining river structure and extracting line segments...")

# Iterate through each River object and its LineString segments
for river_idx, river in enumerate(rivers):
    # river is a River object, convert to list to get LineString segments
    segments = list(river)
    print(f"  River {river_idx}: {len(segments)} LineString segments")
    
    for segment_idx, segment in enumerate(segments):
        # segment is a LineString with multiple coordinate pairs
        coords = list(segment.coords)
        n_line_segs = len(coords) - 1
        
        # Check each line segment (pair of consecutive coordinates)
        for line_seg_idx in range(n_line_segs):
            # Create a LineString from two consecutive points
            line_seg = ShapelyLineString([coords[line_seg_idx], coords[line_seg_idx + 1]])
            
            # Check if this line segment intersects with any river cell
            intersects = False
            for cell_poly in river_cell_polygons:
                if line_seg.intersects(cell_poly):
                    intersects = True
                    break
            
            if intersects:
                line_segments_in_cells.append(line_seg)
                line_segments_in_cells_info.append((river_idx, segment_idx, line_seg_idx))

# Calculate total line segments across all rivers
total_line_segments = 0
for river in rivers:
    for segment in list(river):
        total_line_segments += len(list(segment.coords)) - 1

print(f"\nTotal line segments across all rivers: {total_line_segments}")
print(f"Line segments overlapping with river cells: {len(line_segments_in_cells)}")
print(f"From {len(set([info[0] for info in line_segments_in_cells_info]))} different River objects")
print(f"From {len(set([(info[0], info[1]) for info in line_segments_in_cells_info]))} different LineString segments")

In [ ]:
# Visualize - showing individual line segments in river cells
fig, ax = plt.subplots(figsize=(12, 8))

# Add base mesh
polygons_viz = mc.PolyCollection(polygon_coords, edgecolor='k', cmap='Blues', linewidth=0.5, alpha=0.7)
polygons_viz.set_array(data)
polygons_viz.set_clim(vmin=0, vmax=0.01)
ax.add_collection(polygons_viz)

# Highlight river cells
highlighted_polygons_viz = mc.PolyCollection(highlighted_polygons_coords, 
                                             edgecolor='red', 
                                             facecolors='none', 
                                             linewidth=1,
                                             alpha=0.7)
ax.add_collection(highlighted_polygons_viz)

# Plot all river network (full network in light blue)
watershed_workflow.plot.rivers(rivers, crs_daymet, ax=ax, colors='blue', linewidth=1)

# Plot individual line segments IN river cells (in gold)
for line_seg in line_segments_in_cells:
    coords_seg = np.array(line_seg.coords)
    ax.plot(coords_seg[:, 0], coords_seg[:, 1], color='gold', linewidth=1, zorder=10)

# Set plot limits
# ax.set_xlim(coords[:, 0].min(), coords[:, 0].max())
# ax.set_ylim(coords[:, 1].min(), coords[:, 1].max())

ax.set_aspect('equal')
ax.set_xlabel('X [m]')
ax.set_ylabel('Y [m]')
plt.colorbar(polygons_viz, ax=ax, label='surface-ponded_depth [m]')

plt.title(f'Individual Line Segments in River Cells (100% positive head)\n{len(line_segments_in_cells)} line segments overlap with river cells')

# Add legend
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor='none', edgecolor='red', linewidth=2, label='River cells (100% +head)'),
    Line2D([0], [0], color='gold', linewidth=2.5, label=f'Line segs in cells ({len(line_segments_in_cells)})'),
    Line2D([0], [0], color='lightblue', linewidth=1.5, label=f'All river network')
]
ax.legend(handles=legend_elements, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Save rivers and line segments in river cells for later use
import pickle

# Prepare data to save
river_data = {
    'crs': crs_daymet,  # Save the projection coordinate system
    'all_rivers': rivers,  # Full set of River objects (list of River objects)
    'line_segments_in_cells': line_segments_in_cells,  # Individual line segments (2-point LineStrings) in river cells
    'line_segments_in_cells_info': line_segments_in_cells_info,  # (river_idx, segment_idx, line_seg_idx) tuples
    'river_cell_indices': river_cell_indices,  # Triangle indices with 100% positive head
    'river_cell_polygons': river_cell_polygons,  # Actual polygon geometries
    'positive_percentages': positive_percentages,  # Percentage of positive head for all cells
    'metadata': {
        'total_river_objects': len(rivers),
        'total_linestring_segments': sum(len(list(r)) for r in rivers),
        'total_line_segments': total_line_segments,
        'line_segments_in_cells': len(line_segments_in_cells),
        'unique_rivers_in_cells': len(set([info[0] for info in line_segments_in_cells_info])),
        'unique_linestrings_in_cells': len(set([(info[0], info[1]) for info in line_segments_in_cells_info])),
        'total_cells': n_space,
        'river_cells': len(river_cell_indices),
        'crs_name': 'DayMet CRS (Lambert Conformal Conic)',
        'site_name': site_name
    }
}

# Save as pickle file
output_filename = f'./site_selections/river_cells_and_segments.pkl'
with open(output_filename, 'wb') as f:
    pickle.dump(river_data, f)

print(f"River data saved to: {output_filename}")
print(f"\nSaved data includes:")
print(f"  - CRS: {river_data['metadata']['crs_name']}")
print(f"  - Total River objects: {river_data['metadata']['total_river_objects']}")
print(f"  - Total LineString segments: {river_data['metadata']['total_linestring_segments']}")
print(f"  - Total line segments (coordinate pairs): {river_data['metadata']['total_line_segments']}")
print(f"  - Line segments in river cells: {river_data['metadata']['line_segments_in_cells']}")
print(f"  - From unique River objects: {river_data['metadata']['unique_rivers_in_cells']}")
print(f"  - From unique LineStrings: {river_data['metadata']['unique_linestrings_in_cells']}")
print(f"  - Total surface cells: {river_data['metadata']['total_cells']}")
print(f"  - River cells (100% positive head): {river_data['metadata']['river_cells']}")
print(f"\nData structure:")
print(f"  - line_segments_in_cells: List of 2-point LineStrings")
print(f"  - line_segments_in_cells_info: List of (river_idx, segment_idx, line_seg_idx) tuples")
print(f"\nTo load later:")
print(f"  with open('{output_filename}', 'rb') as f:")
print(f"      river_data = pickle.load(f)")
print(f"  crs_daymet = river_data['crs']")
print(f"  all_rivers = river_data['all_rivers']  # River objects")
print(f"  line_segs_in_cells = river_data['line_segments_in_cells']  # 2-point LineStrings")